# Notebook 05: Baseline Logistic Regression Model

## Purpose
Train a baseline logistic regression model to predict child stunting using the processed modeling dataset.

## Objectives
1. Load the processed dataset  
2. Prepare predictors and target  
3. Encode categorical variables  
4. Scale numeric variables  
5. Train a class-balanced logistic regression model  
6. Evaluate baseline performance  
7. Save the trained model for later use

In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Define project paths
PROJECT_ROOT = Path("/Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
MODELS = OUTPUTS / "models"

# Ensure model folder exists
MODELS.mkdir(parents=True, exist_ok=True)

In [2]:
# Load modeling dataset
df = pd.read_parquet(DATA_PROCESSED / "model_dataset.parquet")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5415, 9)


,v001,v002,hw70,stunted,v012_x,v106_x,v190_x,v025_x,hv009
0,1,9,-0.84,0,27,primary,poorer,rural,4
1,1,24,-2.52,1,40,primary,richer,rural,7
2,1,24,-1.98,0,40,primary,richer,rural,7
3,1,39,-0.67,0,24,primary,richer,rural,3
4,1,69,-2.04,1,29,primary,richer,rural,9


## Prepare target and predictors

In [3]:
# Define target
y = df["stunted"]

# Remove target, outcome source, and identifiers from predictors
X = df.drop(columns=["stunted", "hw70", "v001", "v002"]).copy()

# Clean categorical variables
for col in ["v106_x", "v190_x", "v025_x"]:
    X[col] = X[col].astype(str).str.strip().str.lower()

# Ensure numeric variables are numeric
X["v012_x"] = pd.to_numeric(X["v012_x"], errors="coerce")
X["hv009"] = pd.to_numeric(X["hv009"], errors="coerce")

# Remove any remaining incomplete rows
X = X.dropna()
y = y.loc[X.index]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

X shape: (5415, 5)
y shape: (5415,)


,v012_x,v106_x,v190_x,v025_x,hv009
0,27,primary,poorer,rural,4
1,40,primary,richer,rural,7
2,40,primary,richer,rural,7
3,24,primary,richer,rural,3
4,29,primary,richer,rural,9


## Split data into training and testing sets

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4332, 5)
Test shape: (1083, 5)


In [5]:
# Define categorical and numeric columns
cat_cols = ["v106_x", "v190_x", "v025_x"]
num_cols = ["v012_x", "hv009"]

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

# Build baseline model pipeline
model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            class_weight="balanced"
        ))
    ]
)

## Train the baseline model

In [6]:
# Train model
model.fit(X_train, y_train)

print("Baseline logistic regression model trained successfully")

Baseline logistic regression model trained successfully


In [7]:
# Evaluate baseline performance

# Generate predictions
y_pred = model.predict(X_test)

# Print core metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))

Accuracy: 0.5097
Precision: 0.3652
Recall: 0.5737
F1 Score: 0.4463


/Users/frack/miniforge3/envs/ocr/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frack/miniforge3/envs/ocr/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frack/miniforge3/envs/ocr/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [8]:
# Print detailed classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.48      0.56       710
           1       0.37      0.57      0.45       373

    accuracy                           0.51      1083
   macro avg       0.52      0.52      0.50      1083
weighted avg       0.57      0.51      0.52      1083



## Save trained baseline model

In [9]:
# Save model for reuse and future deployment
model_path = MODELS / "baseline_logistic_model.pkl"
joblib.dump(model, model_path)

print("Saved model to:", model_path)

Saved model to: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/outputs/models/baseline_logistic_model.pkl


## Summary

A class-balanced baseline logistic regression model has been trained and evaluated.

This model provides the first benchmark for predicting child stunting using a small set of demographic, socioeconomic, and household variables.

The trained model has been saved and can later be reused for comparison, inference, and deployment in the web application.